<a href="https://colab.research.google.com/github/ArtyC0des/Downlink-RAN-Scheduler-Simulation/blob/main/RAN_Scheduler_Project_CMSC_491_AdvWireless_Raiyaan_T.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ignore, this is for google drive mount for .dat file access**

In [ ]:
from google.colab import drive
import os

# 1. Mount your Google Drive
# This will trigger a popup asking you to give Colab permission to access your Drive
drive.mount('/content/drive')

# 2. Define the path to your specific folder
# IMPORTANT: Change 'RAN_Project' to the exact name of the folder you created in Step 1
project_folder = '/content/drive/MyDrive/CMPE Senior Spring/CMSC491Wireless/Coding Project/Data & Figures'

# 3. Change Colab's working directory to that folder
try:
    os.chdir(project_folder)
    print(f"Successfully moved to: {os.getcwd()}")
    print("Files currently in this folder:")
    print(os.listdir()) # This should print your two .dat files!
except FileNotFoundError:
    print(f"Error: Could not find the folder {project_folder}. Please check the spelling.")

Mounted at /content/drive
Successfully moved to: /content/drive/MyDrive/CMPE Senior Spring/CMSC491Wireless/Coding Project/Data & Figures
Files currently in this folder:
['PA1_arriving_bits.dat', 'PA1_CQI.dat']


##Simulation program and RR scheduler

In [ ]:
#!/usr/bin/env python3
# RAN scheduler in Python
# Code for students that implements simulated RAN over time.  Students need to code missing parts of main routine, as well as
# scheduling functions RR(), BET(), MT(), PF()
# Last update: Apr 19, 2024 - Nikko

# required numpy library
import csv
import numpy as np
import os

# Constants
CQI_MAX = 15
tmax = 1000
N_UE = 10
N_RB = 12
MAX_LINE_LENGTH = 100
NUM_VALUES = 11

In [ ]:
# Global arrays
# CQI = np.zeros((tmax + 1, N_UE), dtype=int)
# arriving_bits = np.zeros((tmax + 1, N_UE), dtype=int)
BITS_PER_SYMBOL_by_CQI_value = np.array([0, 2, 2, 2, 2, 4, 4, 4, 4, 6, 6, 6, 6, 8, 8, 8])
# queue_length_in_bits = np.array([500] * N_UE)
# sum_queue_length_in_bits = np.zeros(N_UE, dtype=int)
# sum_nbits_sent_so_far = np.zeros(N_UE, dtype=int)


# INIT FUNCTION - load files
def init_load_files(CQI_FILENAME="PA1_CQI.dat", nbit_FILENAME="PA1_arriving_bits.dat"):
    CQI = np.zeros((tmax + 1, N_UE), dtype=int)
    arriving_bits = np.zeros((tmax + 1, N_UE), dtype=int)
    try:
        # Open and read in CQI values
        with open(CQI_FILENAME) as file:
            reader = csv.reader(file)
            for row in reader:
                tval = int(row[0])
                CQI[tval] = row[1:]

        # Open and read in number of bit arrivals per time unit per UE values
        with open(nbit_FILENAME) as file:
            reader = csv.reader(file)
            for row in reader:
                tval = int(row[0])
                arriving_bits[tval] = row[1:]

    except FileNotFoundError as e:
        print(f"Unable to open file: {e}")
        exit()

    except Exception as e:
        print(f"Error reading or processing the files: {e}")
        exit()

    return (CQI, arriving_bits)

def write_csv_results(history_bits_sent, history_rbs_sent, scheduler_name="RR"):
    try:
        out_dir = "graphing"
        if not os.path.exists(out_dir):
            os.makedirs(out_dir)

        filename1 = os.path.join(out_dir, f"{scheduler_name}_bits_sent.csv")
        filename2 = os.path.join(out_dir, f"{scheduler_name}_rbs_sent.csv")

        with open(filename1, 'w', newline='') as f1:
            writer1 = csv.writer(f1)
            writer1.writerow(['time', 'ue_id', 'number_of_bits_sent'])
            for t in range(1, tmax + 1):
                for ue in range(N_UE):
                    writer1.writerow([t, ue, history_bits_sent[t][ue]])

        with open(filename2, 'w', newline='') as f2:
            writer2 = csv.writer(f2)
            writer2.writerow(['time', 'ue_id', 'number_of_ues_sent'])
            for t in range(1, tmax + 1):
                for ue in range(N_UE):
                    writer2.writerow([t, ue, history_rbs_sent[t][ue]])
    except Exception as e:
        print(f"Error writing CSV files: {e}")

In [ ]:
def print_info_before(t, CQI_t, arriving_bits_t, queue_length_in_bits, sum_nbits_sent_so_far):
    print("\nScheduler info at time: " + str(t) + " BEFORE scheduling")
    print("   CQI for each UE queue:           " + " ".join(f"{x:06d}" for x in CQI_t) + " ")
    print("   bits/symbol per UE, for UE CQI:  " + " ".join(f"{BITS_PER_SYMBOL_by_CQI_value[x]:06d}" for x in CQI_t) + " ")
    print("   nbits arriving, per UE queue:    " + " ".join(f"{x:06d}" for x in arriving_bits_t) + " ")
    print("   #bits in UE q (after arrivals):  " + " ".join(f"{x:06d}" for x in queue_length_in_bits) + " ")
    print("   nbits sent so far, per UE queue: " + " ".join(f"{x:06d}" for x in sum_nbits_sent_so_far) + " ")

def print_info_after(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far):
    print("  AFTER scheduling:")
    print("   # RB's assigned to each UE:      " + " ".join(f"{x:06d}" for x in UE_to_RB_count) + " ")
    print("   #bits sent by each UE this time: " + " ".join(f"{x:06d}" for x in UE_to_RB_nbits) + " ")
    print("   #bits in UE q (after send):      " + " ".join(f"{x:06d}" for x in queue_length_in_bits) + " ")
    print("   nbits sent so far, per UE queue: " + " ".join(f"{x:06d}" for x in sum_nbits_sent_so_far) + " ")


In [ ]:
def main():
    # Initialize the simulation

    # load files
    CQI, arriving_bits = init_load_files()

    # current CQI value for UE[0] through UE[9]
    current_CQI = [0] * N_UE
    # current # bits arriving for eventual transmission for UE[0] through UE[9]
    current_nbits_arriving = [0] * N_UE
    # current bits in the queues
    queue_length_in_bits = np.array([500] * N_UE)
    # total bits sent
    sum_nbits_sent_so_far = np.zeros(N_UE, dtype=int)
    # total queue length
    sum_queue_length_in_bits = np.zeros(N_UE, dtype=int)
    # performance stats averaged over all tmx time steps
    avg_queue_length_in_bits = [0] * N_UE
    avg_throughput = [0] * N_UE

    # initialize history arrays for CSV
    history_bits_sent = np.zeros((tmax + 1, N_UE), dtype=int)
    history_rbs_sent = np.zeros((tmax + 1, N_UE), dtype=int)
    # track running sum of RBs assigned just like sum_nbits_sent_so_far
    total_rbs_sent_so_far = np.zeros(N_UE, dtype=int)

    # scheduler name for CSV output (change this to "BET", "MT", or "PF" when implementing them)
    scheduler_name = "PF"

    # Simulation loop for each time step
    for t in range(1, tmax + 1):

        # get CQI values for UE's 0 ... 9 for this time unit into current_QCI[0] ... current_QCI[9]
        for i in range(0, N_UE):
            current_CQI[i] = CQI[t][i]

        # get number of arriving bits for each UE for this time unit E
        for i in range(0, N_UE):
            current_nbits_arriving[i] = arriving_bits[t][i];

        # udpate queue length of unsent bits for each UE by adding in number of current arrived bits
        # update sum of queue lengths over all intervals for each UE, so we can compute average queue length per UE later
        # initialize   UE_to_RB_count[] and UE_to_RB_nbits[], which will be completed by scheduler.
        for i in range(N_UE):
            queue_length_in_bits[i] += current_nbits_arriving[i]
            sum_queue_length_in_bits[i] += queue_length_in_bits[i]


        # UE_to_RB_count[i] in # of RBs assigned to UEi, returned by the scheduler
        UE_to_RB_count = [0] * N_UE
        # UE_to_RB_count[i] in # bits from UEi queue to be tranmsitted in RBs this time unit returned by the scheduler
        UE_to_RB_nbits = [0] * N_UE

        # Optional printing of detailed information before processing
        if t <= 6:
            print_info_before(t, CQI[t], arriving_bits[t], queue_length_in_bits, sum_nbits_sent_so_far)

        ########################################################################
        ## Scheduling: Students will implement RR, BET, MT, or PF scheduling. ##
        ########################################################################

        # change RR to other scheduling.
        if RR(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far, current_CQI) == -1:
            print("panic: RR scheduler returned -1")
            break

        for i in range(0, N_UE):
            # update variables
            queue_length_in_bits[i] -= UE_to_RB_nbits[i]
            sum_nbits_sent_so_far[i] += UE_to_RB_nbits[i]
            total_rbs_sent_so_far[i] += UE_to_RB_count[i]

            # store history for CSV
            history_bits_sent[t][i] = sum_nbits_sent_so_far[i]
            history_rbs_sent[t][i] = total_rbs_sent_so_far[i]
        if t <= 6:
            print_info_after(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far)

    # Compute average metrics after all time steps
    for i in range(N_UE):
        avg_queue_length_in_bits[i] = sum_queue_length_in_bits[i] // tmax
        avg_throughput[i] = sum_nbits_sent_so_far[i] // tmax

    # Printing final average metrics
    print("\n  Average throughput per UE:       " + ' '.join(f"{x:06d}" for x in avg_throughput) + " ")
    print("  Average UE queue length in bits: " + ' '.join(f"{x:06d}" for x in avg_queue_length_in_bits) + " \n")

    # Write CSV files at the end of simulation
    write_csv_results(history_bits_sent, history_rbs_sent, scheduler_name)

########################################################
## Implement your RR, BET, MT, or PF scheduling here. ##
########################################################

# for RR - UE that is currently first/next in Round Robin sequence to get assigned an RB.  GLOBAL variable
RR_current_UE = 0

# implement round robin scheduling at time t, given queue_length_in_bits[i],CQI[t][i] for UEi
# update global vars to reflect  transmitted bits from queuelength count, increase # bits trasnmitted for each UE under RR
def RR(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far, current_CQI):

    global RR_current_UE

    # internal RR use, updated as RBs assigned; main() will update queue_length_in_bits[]
    RR_internal_queue_length_in_bits = np.copy(queue_length_in_bits)

    # assign i_th RB to a UE
    for i in range(N_RB):
        nzerobits = 0
        # find next UE with data bits to be sent, starting at RR_current_UE
        # check RR-internal queue length variable to see if there are enough bits to make at least one symbol for RR_current_UE.
        while RR_internal_queue_length_in_bits[RR_current_UE] < BITS_PER_SYMBOL_by_CQI_value[current_CQI[RR_current_UE]]:
            RR_current_UE = (RR_current_UE + 1) % N_UE
            nzerobits += 1
            # no data to send at all - if all queues empty there's a problem!
            if nzerobits > N_UE:
                return -1

        # get CQI for this UE
        current_UE_CQI = current_CQI[RR_current_UE]
        # get # bits per symbol for this UE, given CQI for this UE
        bits_per_symbol = BITS_PER_SYMBOL_by_CQI_value[current_UE_CQI]
        # current queue length for this UE in symbols
        nsymbols = RR_internal_queue_length_in_bits[RR_current_UE] // bits_per_symbol
        # if more than 84 symbols worth of data in queue, fill entire RB with 84 symbols
        nsymbols_sent = min(nsymbols, 84)

        # total bits to sent
        nbits_sent = nsymbols_sent * bits_per_symbol

        # increase total bits sent to this UE
        UE_to_RB_nbits[RR_current_UE] += nbits_sent
        # increase total RB counter to this UE
        UE_to_RB_count[RR_current_UE] += 1

        # update internal QL
        RR_internal_queue_length_in_bits[RR_current_UE] -= nbits_sent;

        # move to the next UE
        RR_current_UE = (RR_current_UE + 1) % N_UE

    return 0

# implement blind equal throughput (BET) scheduling at time t, given queue_length_in_bits[i],current_CQI[i]
# and any other variables or data strcutures you need to adding

# Global Variable to track average past throughput for BET scheduler.
# needs to be inialized to 1.0 for all UEs as per instructions
BET_past_throughput = np.ones(N_UE, dtype=float)

def BET(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far, current_CQI):

    # bring in global & set moving average weight
    global BET_past_throughput
    beta = 0.125

    # Placeholder for the Blind Equal Throughput scheduling function
    BET_internal_queue_length_in_bits = np.copy(queue_length_in_bits)

    # Rank the UEs based on past throughput R^i(t-1)
    # I will use python sorted because it is stable and will
    # handle tie prio collisons
    ue_priority = sorted(range(N_UE), key=lambda x: BET_past_throughput[x])

    rbs_assigned = 0

    # now assign RBs
    for ue in ue_priority:

      # checks to see if all RBs are assigned
      if rbs_assigned >= N_RB:
        break

      # current_UE_CQI = current
      current_CQI = current_CQI[ue]
      bits_per_symbol = BITS_PER_SYMBOL_by_CQI_value[current_CQI]

      # if CQI = 0 no data can be sent through this UE so lets get rid of it
      if bits_per_symbol == 0:
        continue

      # send as many bits as we can
      while rbs_assigned < N_RB and BET_internal_queue_length_in_bits[ue] >= bits_per_symbol:
        nsymbols = BET_internal_queue_length_in_bits[ue] // bits_per_symbol
        nsymbosl_sent = min(nsymbols, 84) # for 84 max symbols
        nbits_sent = nsymbosl_sent * bits_per_symbol

        # update RBs and bit counts for this UE
        UE_to_RB_nbits[ue] += nbits_sent
        UE_to_RB_count[ue] += 1
        BET_internal_queue_length_in_bits[ue] -= nbits_sent

        rbs_assigned += 1

    # update historical average
    for ue in range(N_UE):
      current_throughput = UE_to_RB_nbits[ue]
      BET_past_throughput[ue] = (1.0 - beta) * BET_past_throughput[ue] + beta * current_throughput


    # done!
    return 0

# implement maximum throughput (MT) scheduling at time t, given queue_length_in_bits[i],current_CQI[i]
# and any other variables or data strcutures you need to adding
def MT(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far, current_CQI):

    # local variable
    MT_internal_queue_length_in_bits = np.copy(queue_length_in_bits)

    # Sort all the UE's (we want the best of the best @ the top)
    ue_priority = sorted(range(N_UE), key=lambda x: (-current_CQI[x], x))

    rbs_assigned = 0

    # allocate RBs
    for ue in ue_priority:

      # checks to see if all RBs are assigned
      if rbs_assigned >= N_RB:
        break

      # current_UE_CQI = current
      current_UE_CQI = current_CQI[ue]
      bits_per_symbol = BITS_PER_SYMBOL_by_CQI_value[current_UE_CQI]

      # send as many bits as we can
      while rbs_assigned < N_RB and MT_internal_queue_length_in_bits[ue] >= bits_per_symbol:
        nsymbols = MT_internal_queue_length_in_bits[ue] // bits_per_symbol
        nsymbosl_sent = min(nsymbols, 84) # for 84 max symbols
        nbits_sent = nsymbosl_sent * bits_per_symbol

        # update RBs and bit counts for this UE
        UE_to_RB_nbits[ue] += nbits_sent
        UE_to_RB_count[ue] += 1
        MT_internal_queue_length_in_bits[ue] -= nbits_sent

        rbs_assigned += 1


    return 0

# implement proportionally fair (PF) scheduling at time t, given queue_length_in_bits[i],current_CQI[i]
# and any other variables or data strcutures you need to adding

# Global Variable to track average past throughput for PF scheduler.
# needs to be inialized to 1.0 for all UEs as per instructions
PF_past_throughput = np.ones(N_UE, dtype=float)

def PF(UE_to_RB_count, UE_to_RB_nbits, queue_length_in_bits, sum_nbits_sent_so_far, current_CQI):

    # intialzie global variable
    global PF_past_throughput

    # The exponential moving average weight beta
    beta = 0.125

    # Placeholder for the Proportional Fair scheduling function
    PF_internal_queue_length_in_bits = np.copy(queue_length_in_bits)

    # calculate PF metric
    pf_metric = np.zeros(N_UE, dtype=float)

    for ue in range(N_UE):

      # instant rate calc
      bits_per_symbol = BITS_PER_SYMBOL_by_CQI_value[current_CQI[ue]]
      ins_rate = bits_per_symbol * 84

      # PF ratio calculation IMPORANT
      pf_metric[ue] = ins_rate / PF_past_throughput[ue]

      # Rank UEs based on pf metric
      ue_priority = sorted(range(N_UE), key=lambda x: (-pf_metric[x], x))


      rbs_assigned = 0

      #allocate RBs
      for ue in ue_priority:

        # checks to see if all RBs are assigned
        if rbs_assigned >= N_RB:
          break

        current_UE_CQI = current_CQI[ue]
        bits_per_symbol = BITS_PER_SYMBOL_by_CQI_value[current_UE_CQI]

        if bits_per_symbol == 0:
          continue

        # pack it
        while rbs_assigned < N_RB and PF_internal_queue_length_in_bits[ue] >= bits_per_symbol:
          nsymbols = PF_internal_queue_length_in_bits[ue] // bits_per_symbol
          nsymbosl_sent = min(nsymbols, 84) # for 84 max symbols
          nbits_sent = nsymbosl_sent * bits_per_symbol

          # update RBs bits
          UE_to_RB_nbits[ue] += nbits_sent
          UE_to_RB_count[ue] += 1
          PF_internal_queue_length_in_bits[ue] -= nbits_sent

          rbs_assigned += 1

      # update historical average
      for ue in range(N_UE):
        current_throughput = UE_to_RB_nbits[ue]
        PF_past_throughput[ue] = (1.0 - beta) * PF_past_throughput[ue] + beta * current_throughput


    return 0

In [ ]:
if __name__ == '__main__':
    # start main function
    main()


Scheduler info at time: 1 BEFORE scheduling
   CQI for each UE queue:           000006 000004 000006 000006 000004 000004 000004 000005 000003 000005 
   bits/symbol per UE, for UE CQI:  000004 000002 000004 000004 000002 000002 000002 000004 000002 000004 
   nbits arriving, per UE queue:    000734 000711 000781 000410 000358 000306 000423 000379 000381 000590 
   #bits in UE q (after arrivals):  001234 001211 001281 000910 000858 000806 000923 000879 000881 001090 
   nbits sent so far, per UE queue: 000000 000000 000000 000000 000000 000000 000000 000000 000000 000000 
  AFTER scheduling:
   # RB's assigned to each UE:      000002 000002 000001 000001 000001 000001 000001 000001 000001 000001 
   #bits sent by each UE this time: 000672 000336 000336 000336 000168 000168 000168 000336 000168 000336 
   #bits in UE q (after send):      000562 000875 000945 000574 000690 000638 000755 000543 000713 000754 
   nbits sent so far, per UE queue: 000672 000336 000336 000336 000168 000168 0

## Graph generating Script

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_scheduler_graphs(schedulers=['RR', 'BET', 'MT', 'PF']):
    """
    Reads the output CSVs for multiple schedulers and generates two combined
    line charts (bits sent and RBs sent) for each available scheduler.
    Ignores schedulers whose CSV output files are missing.
    """
    bits_data = {}
    rbs_data = {}
    out_dir = "graphing"

    for scheduler in schedulers:
        bits_file = os.path.join(out_dir, f"{scheduler}_bits_sent.csv")
        rbs_file = os.path.join(out_dir, f"{scheduler}_rbs_sent.csv")

        if os.path.exists(bits_file):
            print(f"Loading {bits_file}...")
            df_bits = pd.read_csv(bits_file)
            # Pivot table to make X-axis 'time', Y-axis 'bits_sent', and separate lines per 'ue_id'
            bits_data[scheduler] = df_bits.pivot(index='time', columns='ue_id', values='number_of_bits_sent')
        else:
            print(f"Skipping {bits_file} (not found).")

        if os.path.exists(rbs_file):
            print(f"Loading {rbs_file}...")
            df_rbs = pd.read_csv(rbs_file)
            rbs_data[scheduler] = df_rbs.pivot(index='time', columns='ue_id', values='number_of_ues_sent')
        else:
            print(f"Skipping {rbs_file} (not found).")

    # Plot 1: Number of Bits Sent per UE over time
    for scheduler, data in bits_data.items():
        plt.figure(figsize=(12, 6))
        for ue_col in data.columns:
            plt.plot(data.index, data[ue_col], label=f'UE {ue_col}')
        plt.title(f'[{scheduler}] Number of Bits Sent per UE Over Time')
        plt.xlabel('Time')
        plt.ylabel('Number of Bits Sent')
        plt.legend(title='UE ID', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.tight_layout()
        out_name = os.path.join(out_dir, f"{scheduler}_bits_sent_graph.png")
        plt.savefig(out_name)
        print(f"Saved graph: {out_name}")
        plt.close()

    # Plot 2: Number of RBs Sent per UE over time
    for scheduler, data in rbs_data.items():
        plt.figure(figsize=(12, 6))
        for ue_col in data.columns:
            plt.plot(data.index, data[ue_col], label=f'UE {ue_col}')
        plt.title(f'[{scheduler}] Number of RBs Sent per UE Over Time')
        plt.xlabel('Time')
        plt.ylabel('Number of RBs Sent')
        plt.legend(title='UE ID', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.tight_layout()
        out_name = os.path.join(out_dir, f"{scheduler}_rbs_sent_graph.png")
        plt.savefig(out_name)
        print(f"Saved graph: {out_name}")
        plt.close()

if __name__ == "__main__":
    plot_scheduler_graphs()

Loading graphing/RR_bits_sent.csv...
Loading graphing/RR_rbs_sent.csv...
Loading graphing/BET_bits_sent.csv...
Loading graphing/BET_rbs_sent.csv...
Loading graphing/MT_bits_sent.csv...
Loading graphing/MT_rbs_sent.csv...
Loading graphing/PF_bits_sent.csv...
Loading graphing/PF_rbs_sent.csv...
Saved graph: graphing/RR_bits_sent_graph.png
Saved graph: graphing/BET_bits_sent_graph.png
Saved graph: graphing/MT_bits_sent_graph.png
Saved graph: graphing/PF_bits_sent_graph.png
Saved graph: graphing/RR_rbs_sent_graph.png
Saved graph: graphing/BET_rbs_sent_graph.png
Saved graph: graphing/MT_rbs_sent_graph.png
Saved graph: graphing/PF_rbs_sent_graph.png
